# 02 — Core Database Setup (UDA-Hub)

Initialises the **core** SQLite database that backs UDA-Hub itself, and
loads the Knowledge base from `data/external/cultpass_articles.jsonl`.

Required tables (per rubric): `Account`, `User`, `Ticket`, `TicketMetadata`,
`TicketMessage`, `Knowledge`.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from data.core import db, seed
from agentic.config import settings
settings.core_db_path

## 1. (Re)create the database and load seed rows

In [ ]:
counts = seed.seed_all(reset=True)
counts

## 2. Verify required tables

Each table must exist and have at least one row.

In [ ]:
from tabulate import tabulate
for tbl in ['Account','User','Ticket','TicketMetadata','TicketMessage','Knowledge']:
    n = db.fetch_one(f'SELECT COUNT(*) AS n FROM {tbl}')['n']
    print(f'{tbl:<16} {n}')

## 3. Knowledge base — categories and counts

In [ ]:
kb = db.fetch_all('SELECT category, COUNT(*) AS n FROM Knowledge GROUP BY category ORDER BY category')
print(tabulate(kb, headers='keys'))
total = db.fetch_one('SELECT COUNT(*) AS n FROM Knowledge')['n']
print(f'\nTotal articles: {total}  (rubric requires \u226514)')

In [ ]:
for row in db.fetch_all('SELECT article_id, title, category FROM Knowledge ORDER BY article_id'):
    print(f"{row['article_id']}  [{row['category']:<10}] {row['title']}")

## 4. Spot-check a single article

In [ ]:
art = db.fetch_one('SELECT * FROM Knowledge WHERE article_id=?', ('cp_kb_013',))
print(art['title']); print('-'*60); print(art['body'])

## 5. Cross-DB linkage — Account.external_member_id → CultPassMember

Every Account in the core DB is linked to a CultPass member id, which is
what the `cultpass_member_lookup` tool uses to join.

In [ ]:
print(tabulate(db.fetch_all('''
    SELECT account_id, name, plan, status, external_member_id
      FROM Account ORDER BY account_id
'''), headers='keys'))

## 6. Build the FAISS vector index over Knowledge

In [ ]:
import os
if os.getenv('OPENAI_API_KEY'):
    from agentic.retrieval import build_or_load_vectorstore
    vs = build_or_load_vectorstore(rebuild=True)
    print('FAISS index built with', vs.index.ntotal, 'vectors')
else:
    print('OPENAI_API_KEY not set — skipping FAISS index build (keyword fallback will be used at runtime).')

---

Core database is ready. Continue to `03_agentic_app.ipynb` to run the workflow.